<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment1/DNNAssignment1Vgg19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
import sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers
import numpy as np

In [ ]:
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()

train_filter = (y_train_tmp < 20).flatten()
test_filter = (y_test_tmp < 20).flatten()

x_train = x_train_tmp[train_filter]
y_train = y_train_tmp[train_filter]
x_test = x_test_tmp[test_filter]
y_test = y_test_tmp[test_filter]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

trainY = to_categorical(y_train, num_classes=20)
testY = to_categorical(y_test, num_classes=20)

In [ ]:
input_shape = (32, 32, 3)

vgg19_model = keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
    pooling='avg'
)
model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        vgg19_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dropout(0.4),
        layers.Dense(127),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)                   │ (None, 512)                 │      20,024,384 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │         262,656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 127)                 │          65,151 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 127)                 │             508 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 127)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 20)                  │           2,560 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 20,357,307 (77.66 MB)

 Trainable params: 20,356,029 (77.65 MB)

 Non-trainable params: 1,278 (4.99 KB)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

epochs = 10
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy',],
)

model.fit(x_train, trainY, epochs=epochs, callbacks=[early_stopping], validation_split=0.1)
model.save('vgg19_model.keras')

Epoch 1/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 45s 88ms/step - accuracy: 0.0570 - loss: 3.1946 - val_accuracy: 0.0500 - val_loss: 3.1002
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 23s 56ms/step - accuracy: 0.1045 - loss: 2.8177 - val_accuracy: 0.0990 - val_loss: 2.8187
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 16s 57ms/step - accuracy: 0.1242 - loss: 2.6871 - val_accuracy: 0.1050 - val_loss: 2.7529
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 57ms/step - accuracy: 0.1426 - loss: 2.6205 - val_accuracy: 0.1170 - val_loss: 2.7014
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 58ms/step - accuracy: 0.1556 - loss: 2.5531 - val_accuracy: 0.1050 - val_loss: 2.8175
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 58ms/step - accuracy: 0.1638 - loss: 2.5010 - val_accuracy: 0.1880 - val_loss: 2.4552
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 17s 59ms/step - accuracy: 0.1775 - loss: 2.4363 - val_accuracy: 0.1580 - val_loss: 2.5447
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 17s 59ms/step - accuracy: 0.1955 - loss: 2.3533 - 

In [ ]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.1886 - loss: 2.3210


[2.3217244148254395, 0.1979999989271164]